# 01 - Data understanding

Inspect the actual Figshare workbook: schema, types, missingness, loan statuses and how they map
to the modelling target. Run `python scripts/inspect_data.py` first (it builds the Parquet cache).

> Notebooks display **aggregates only** (no individual loan records) and reuse functions from `src/creditiq/`. Educational project - not for real lending decisions.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from creditiq.config import load_config, setup_logging
from creditiq.cli import processed_path, clean_all_path
from creditiq.reporting import figures as figs

setup_logging("WARNING")
cfg = load_config()
figs.set_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
TARGET = cfg["target"]["name"]

In [2]:
from creditiq.data.load import load_raw, file_digest, list_excel_sheets
from creditiq.data.validate import data_quality_summary
from creditiq.data.target import status_summary
from creditiq.features.leakage_checks import uncatalogued_columns

raw_path = cfg.path("raw_file")
print("Source:", raw_path.name, "| sheets:", list_excel_sheets(raw_path))
print("MD5:", file_digest(raw_path, "md5"), "(Figshare computed_md5: 244d12efa7b8c845263bf26b0d49ae7f)")
raw = load_raw(raw_path, cfg.path("interim_dir"), sheet=cfg["data"]["excel_sheet"])
print(f"Rows x columns: {raw.shape[0]:,} x {raw.shape[1]}")

Source: 12_13_14 Excel.xlsx | sheets: ['2012_13_14']


MD5: 244d12efa7b8c845263bf26b0d49ae7f (Figshare computed_md5: 244d12efa7b8c845263bf26b0d49ae7f)


Rows x columns: 423,810 x 51


## Column profile (types, missingness, cardinality)

In [3]:
quality = data_quality_summary(raw)
display(quality[["dtype", "n_missing", "pct_missing", "n_unique", "min", "max"]])
print("Columns without availability classification:", uncatalogued_columns(raw.columns, cfg["feature_catalog"]))
print("Missing required columns:", [c for c in cfg["data"]["required_columns"] if c not in raw.columns])

,dtype,n_missing,pct_missing,n_unique,min,max
column,,,,,,
loan_amnt,int64,0,0.00,1339,1000,35000
funded_amnt,int64,0,0.00,1339,1000,35000
funded_amnt_inv,float64,0,0.00,1723,950.0,35000.0
term,str,0,0.00,2,None,None
int_rate,float64,0,0.00,188,0.06,0.2606
installment,float64,0,0.00,49362,4.93,1409.99
grade,str,0,0.00,7,None,None
sub_grade,str,0,0.00,35,None,None
emp_title,string,24974,5.89,182439,None,None


Columns without availability classification: []
Missing required columns: []


## Loan statuses and target mapping

`Current`, `Late` and `In Grace Period` loans have **not resolved**; labelling them as good would
bias the target. They are excluded. `Default` (121+ days past due, not yet charged off) is adverse.

In [4]:
status = status_summary(raw, cfg["target"])
display(status)

,n_loans,share_pct,target_class
loan_status,,,
Fully Paid,340444,80.33,negative
Charged Off,70829,16.71,positive (adverse)
Current,11925,2.81,excluded (unresolved)
Late (31-120 days),337,0.08,excluded (unresolved)
In Grace Period,201,0.05,excluded (unresolved)
Late (16-30 days),73,0.02,excluded (unresolved)
Default,1,0.0,positive (adverse)


## Is a fixed-horizon target possible? Resolution by vintage and term

In [5]:
res = raw.assign(
    issue_q=pd.to_datetime(raw["issue_d"]).dt.to_period("Q").astype(str),
    resolved=raw["loan_status"].isin(cfg["target"]["positive_statuses"] + cfg["target"]["negative_statuses"]),
)
tab = res.pivot_table(index="issue_q", columns="term", values="resolved", aggfunc="mean")
display((1 - tab).style.format("{:.1%}").set_caption("Share UNRESOLVED at snapshot"))
print("Latest payment / bureau-pull dates in file:", raw["last_pymnt_d"].max(), raw["last_credit_pull_d"].max())
print("Columns containing a charge-off date:", [c for c in raw.columns if "charge" in c or "default_d" in c])

term,36 months,60 months
issue_q,,
2012Q1,0.0%,0.0%
2012Q2,0.0%,0.0%
2012Q3,0.0%,0.0%
2012Q4,0.0%,0.0%
2013Q1,0.0%,0.0%
2013Q2,0.0%,0.0%
2013Q3,0.0%,0.0%
2013Q4,0.0%,0.1%
2014Q1,0.0%,3.0%


Latest payment / bureau-pull dates in file: 2019-03-01 00:00:00 2019-03-01 00:00:00
Columns containing a charge-off date: []


**Conclusion (derived from the cells above):** the export contains no charge-off date, so a
consistent fixed-horizon (e.g. 12-month) default flag cannot be constructed. The target is the
*lifetime outcome observed at the snapshot among resolved loans*. Unresolved loans are
concentrated in 2014 60-month vintages, which introduces exclusion bias there; a 36-month-only
robustness check is reported in notebook 06.